In [12]:
import os
import numpy as np
import scipy.io as sio
from scipy.signal import butter, filtfilt

# HYPERPARAMETERS & CONFIGURATION
RAW_DATA_DIR = r"C:\Users\risha\OneDrive\Documents\psychopy\BrainInference\eeg_raw_data" 
SAVE_DIR = "./SEED_IV_FEATURES"
FS = 200  
WINDOW_SEC = 4.0
STEP_SEC = 2.0
WINDOW_SAMPLES = int(WINDOW_SEC * FS)
STEP_SAMPLES = int(STEP_SEC * FS)

POWER_GATE_THRESHOLD = 5000.0  

BANDS = {
    'Delta': (1.0, 4.0),
    'Theta': (4.0, 8.0),
    'Alpha': (8.0, 13.0),
    'Beta': (13.0, 31.0),
    'Gamma': (31.0, 50.0)
}

SESSION_LABELS = {
    '1': [1,2,3,0,2,0,0,1,0,1,2,1,1,1,2,3,2,2,3,3,0,3,0,3],
    '2': [2,1,3,0,0,2,0,2,3,3,2,3,2,0,1,1,2,1,0,3,0,1,3,1],
    '3': [1,2,2,1,3,3,3,1,1,2,1,0,2,3,3,0,2,3,0,0,2,0,1,0]
}

if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)

# SIGNAL PROCESSING
def bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data, axis=-1)

def extract_industrial_features(epoch_data):
    features = []
    
    # Standard DE
    for band_name, (low, high) in BANDS.items():
        band_data = bandpass_filter(epoch_data, low, high, FS)
        variance = np.var(band_data, axis=-1)
        de = 0.5 * np.log(2 * np.pi * np.e * variance + 1e-9) 
        features.extend(de)
        
    # MTI 
    mti_data = bandpass_filter(epoch_data[:10, :], 35.0, 50.0, FS)
    mti_power = np.var(mti_data, axis=-1)
    features.extend(mti_power)
        
    return np.array(features)

# MAIN EXTRACTION LOOP
def extract_seed_iv_industrial():
    print("Initiating SEED-IV Industrial Feature Extraction")
    
    for sub_id in range(1, 16):
        sub_X = []
        sub_y = []
        dropped_epochs = 0
        total_epochs = 0
        
        print(f"\nProcessing Subject {sub_id}...")
        
        sub_files = [f for f in os.listdir(RAW_DATA_DIR) if f.startswith(f"{sub_id}_") and f.endswith(".mat")]
        sub_files.sort()
        
        if len(sub_files) == 0:
            print(f"  -> No data found for Subject {sub_id}. Skipping.")
            continue
            
        for session_idx, file_name in enumerate(sub_files):
            session_str = str(session_idx + 1)
            
            if session_str not in SESSION_LABELS:
                continue
                
            current_session_labels = SESSION_LABELS[session_str]
            mat_path = os.path.join(RAW_DATA_DIR, file_name)
            
            try:
                mat_data = sio.loadmat(mat_path)
            except Exception as e:
                print(f"  -> Error loading {file_name}: {e}")
                continue
            
            # --- THE FIX: DYNAMIC KEY SEARCH ---
            for trial_idx in range(1, 25):
                target_suffix = f"eeg{trial_idx}"
                
                # Find whichever variable in the file ends with eeg1, eeg2, etc.
                matching_keys = [k for k in mat_data.keys() if k.endswith(target_suffix)]
                
                if not matching_keys:
                    continue
                    
                trial_key = matching_keys[0] 
                raw_trial = mat_data[trial_key]
                emotion_label = current_session_labels[trial_idx - 1]
                
                total_samples = raw_trial.shape[1]
                for start in range(0, total_samples - WINDOW_SAMPLES + 1, STEP_SAMPLES):
                    end = start + WINDOW_SAMPLES
                    epoch = raw_trial[:, start:end]
                    total_epochs += 1
                    
                    if np.max(np.var(epoch, axis=-1)) > POWER_GATE_THRESHOLD:
                        dropped_epochs += 1
                        continue 
                    
                    features = extract_industrial_features(epoch)
                    
                    sub_X.append(features)
                    sub_y.append(emotion_label)
                    
        if len(sub_X) > 0:
            X_arr = np.array(sub_X)
            y_arr = np.array(sub_y)
            np.save(os.path.join(SAVE_DIR, f"Sub_{sub_id}_X.npy"), X_arr)
            np.save(os.path.join(SAVE_DIR, f"Sub_{sub_id}_y.npy"), y_arr)
            
            retention_rate = ((total_epochs - dropped_epochs) / total_epochs) * 100
            print(f"  -> Extracted {X_arr.shape[0]} clean epochs. (Shape: {X_arr.shape})")
            print(f"  -> Power Gating dropped {dropped_epochs} noisy epochs. (Retention: {retention_rate:.1f}%)")

    print("\nStage 1 Complete. SEED-IV Data Ready for Modeling")

if __name__ == "__main__":
    extract_seed_iv_industrial()

Initiating SEED-IV Industrial Feature Extraction

Processing Subject 1...
  -> Extracted 4590 clean epochs. (Shape: (4590, 320))
  -> Power Gating dropped 381 noisy epochs. (Retention: 92.3%)

Processing Subject 2...
  -> Extracted 3305 clean epochs. (Shape: (3305, 320))
  -> Power Gating dropped 1666 noisy epochs. (Retention: 66.5%)

Processing Subject 3...
  -> Extracted 4877 clean epochs. (Shape: (4877, 320))
  -> Power Gating dropped 94 noisy epochs. (Retention: 98.1%)

Processing Subject 4...
  -> Extracted 4286 clean epochs. (Shape: (4286, 320))
  -> Power Gating dropped 685 noisy epochs. (Retention: 86.2%)

Processing Subject 5...
  -> Extracted 4162 clean epochs. (Shape: (4162, 320))
  -> Power Gating dropped 809 noisy epochs. (Retention: 83.7%)

Processing Subject 6...
  -> Extracted 3403 clean epochs. (Shape: (3403, 320))
  -> Power Gating dropped 1568 noisy epochs. (Retention: 68.5%)

Processing Subject 7...
  -> Extracted 4747 clean epochs. (Shape: (4747, 320))
  -> Power G

In [13]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb

# 1. SEED-IV HYBRID CAE (4-CLASS ARCHITECTURE)
class MultiClassHelperCAE(nn.Module):
    def __init__(self, input_features=320, bottleneck_dim=64, num_classes=4):
        super(MultiClassHelperCAE, self).__init__()
        
        # ENCODER: 320 -> 160 -> 80 -> 40
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.2),
            
            nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),
            
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(0.2),
            
            nn.Flatten(),
            nn.Linear(64 * 40, bottleneck_dim) # 40 comes from the Conv1D math reduction
        )
        
        # DECODER: 40 -> 80 -> 160 -> 320
        self.decoder_fc = nn.Sequential(
            nn.Linear(bottleneck_dim, 64 * 40),
            nn.LeakyReLU(0.2)
        )
        
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(0.2),
            
            nn.ConvTranspose1d(32, 16, kernel_size=5, stride=2, padding=2, output_padding=1),
            nn.BatchNorm1d(16),
            nn.LeakyReLU(0.2),
            
            nn.ConvTranspose1d(16, 1, kernel_size=5, stride=2, padding=2, output_padding=1)
        )
        
        # MULTI-CLASS MLP HEAD (Outputs 4 Raw Logits)
        self.classifier = nn.Sequential(
            nn.Linear(bottleneck_dim, 32),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
            # No Softmax here because PyTorch's CrossEntropyLoss handles it internally
        )

    def forward(self, x):
        latent = self.encoder(x)
        
        dec_x = self.decoder_fc(latent)
        dec_x = dec_x.view(-1, 64, 40) 
        reconstructed_x = self.decoder_conv(dec_x)
        
        logits = self.classifier(latent)
        return reconstructed_x, logits

# 2. MULTI-CLASS DUAL LOSS FUNCTION
def seed_iv_hybrid_loss(recon_x, orig_x, logits, targets, alpha=0.2):
    mse = nn.MSELoss()(recon_x, orig_x)
    # Target must be a long integer for Cross Entropy (Class indices 0,1,2,3)
    ce = nn.CrossEntropyLoss()(logits, targets.long()) 
    
    total_loss = (alpha * mse) + ((1 - alpha) * ce)
    return total_loss, mse, ce

# 3. THE MULTI-CLASS FUSION BENCHMARK
def run_seed_iv_stratified_fusion(total_subjects=15, epochs=30, batch_size=32, w_xgb=0.85, w_cae=0.15):
    print(f"\n=== SEED-IV Multi-Class Stratified Fusion (XGB: {w_xgb} | CAE: {w_cae}) ===")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Hardware utilized: {device}")
    
    SAVE_DIR = "./SEED_IV_FEATURES"
    global_subject_accuracies = []
    
    # Notice the XGBoost parameters changed for Multi-Class
    xgb_base = xgb.XGBClassifier(
        learning_rate=0.05, 
        max_depth=4, 
        n_estimators=50, 
        objective='multi:softprob', 
        num_class=4, # Crucial for SEED-IV
        eval_metric='mlogloss', 
        random_state=42, 
        n_jobs=-1
    )
    
    for sub_id in range(1, total_subjects + 1):
        try:
            X_raw = np.load(os.path.join(SAVE_DIR, f"Sub_{sub_id}_X.npy"))
            y_raw = np.load(os.path.join(SAVE_DIR, f"Sub_{sub_id}_y.npy"))
            
            scaler = RobustScaler()
            X_scaled = scaler.fit_transform(X_raw)
            
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            fold_accs = []
            
            for fold, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y_raw)):
                X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
                y_train, y_test = y_raw[train_idx], y_raw[test_idx]
                
                # --- STREAM A: XGBoost (Multi-Class) ---
                xgb_base.fit(X_train, y_train)
                # Output shape will be (Batch_Size, 4)
                xgb_probs = xgb_base.predict_proba(X_test)
                
                # --- STREAM B: PyTorch CAE (Multi-Class) ---
                X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
                y_train_t = torch.tensor(y_train, dtype=torch.long) # Targets must be long integers
                X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).to(device)
                
                train_dataset = TensorDataset(X_train_t, y_train_t)
                train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
                
                cae_model = MultiClassHelperCAE().to(device)
                optimizer = optim.Adam(cae_model.parameters(), lr=0.001, weight_decay=1e-5)
                
                cae_model.train()
                for epoch in range(epochs):
                    for batch_X, batch_y in train_loader:
                        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                        optimizer.zero_grad()
                        recon_x, logits = cae_model(batch_X)
                        loss, _, _ = seed_iv_hybrid_loss(recon_x, batch_X, logits, batch_y, alpha=0.2)
                        loss.backward()
                        optimizer.step()
                        
                cae_model.eval()
                with torch.no_grad():
                    _, test_logits = cae_model(X_test_t)
                    # Convert raw logits to probabilities
                    cae_probs = F.softmax(test_logits, dim=1).cpu().numpy()
                    
                # --- THE MULTI-CLASS FUSION ENGINE ---
                # Blending two (Batch, 4) probability matrices
                final_probs = (w_xgb * xgb_probs) + (w_cae * cae_probs)
                
                # The prediction is simply the class index (0, 1, 2, or 3) with the highest probability
                final_preds = np.argmax(final_probs, axis=1)
                
                acc = accuracy_score(y_test, final_preds) * 100
                fold_accs.append(acc)
                
            sub_mean_acc = np.mean(fold_accs)
            global_subject_accuracies.append(sub_mean_acc)
            print(f"Subject {sub_id} SEED-IV Hybrid 5-Fold Accuracy: {sub_mean_acc:.2f}%")
                
        except FileNotFoundError:
            print(f"Skipping Subject {sub_id} - Data not found.")
            
    global_acc = np.mean(global_subject_accuracies)
    global_std = np.std(global_subject_accuracies)
    
    print("FINAL SEED-IV HYBRID INTRA-SUBJECT CEILING")
    print(f"Global Average Accuracy: {global_acc:.2f}% (±{global_std:.2f}%)")

if __name__ == "__main__":
    # Ready to fire once Stage 1 completes!
    run_seed_iv_stratified_fusion(total_subjects=15)


=== SEED-IV Multi-Class Stratified Fusion (XGB: 0.85 | CAE: 0.15) ===
Hardware utilized: cuda
Subject 1 SEED-IV Hybrid 5-Fold Accuracy: 81.72%
Subject 2 SEED-IV Hybrid 5-Fold Accuracy: 89.11%
Subject 3 SEED-IV Hybrid 5-Fold Accuracy: 84.42%
Subject 4 SEED-IV Hybrid 5-Fold Accuracy: 90.97%
Subject 5 SEED-IV Hybrid 5-Fold Accuracy: 81.31%
Subject 6 SEED-IV Hybrid 5-Fold Accuracy: 89.98%
Subject 7 SEED-IV Hybrid 5-Fold Accuracy: 86.98%
Subject 8 SEED-IV Hybrid 5-Fold Accuracy: 84.83%
Subject 9 SEED-IV Hybrid 5-Fold Accuracy: 84.07%
Subject 10 SEED-IV Hybrid 5-Fold Accuracy: 82.88%
Subject 11 SEED-IV Hybrid 5-Fold Accuracy: 79.28%
Subject 12 SEED-IV Hybrid 5-Fold Accuracy: 83.30%
Subject 13 SEED-IV Hybrid 5-Fold Accuracy: 88.76%
Subject 14 SEED-IV Hybrid 5-Fold Accuracy: 86.81%
Subject 15 SEED-IV Hybrid 5-Fold Accuracy: 94.54%
FINAL SEED-IV HYBRID INTRA-SUBJECT CEILING
Global Average Accuracy: 85.93% (±4.02%)


In [ ]:
import os
import numpy as np
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import RobustScaler

def run_seed_iv_grid_search(calibration_subject=1):
    print(f"\nSEED-IV Multi-Class XGBoost Grid Search (Subject {calibration_subject})")
    SAVE_DIR = "./SEED_IV_FEATURES"
    
    try:
        X_raw = np.load(os.path.join(SAVE_DIR, f"Sub_{calibration_subject}_X.npy"))
        y_raw = np.load(os.path.join(SAVE_DIR, f"Sub_{calibration_subject}_y.npy"))
    except FileNotFoundError:
        print(f"Error: Could not find data for Subject {calibration_subject}.")
        return

    # Scale the 320 features
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X_raw)
    
    # 1. Define the Multi-Class Grid
    # - Deeper trees (6, 8) help map 4 distinct classes.
    # - Higher n_estimators (100, 200) gives the model more time to find boundaries.
    param_grid = {
        'max_depth': [4, 6, 8],
        'learning_rate': [0.01, 0.05, 0.1],
        'n_estimators': [50, 100, 200]
    }
    
    # 2. Base Model (Configured for 4 classes)
    xgb_base = xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=4,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    )
    
    # 3. The Search Engine (3-Fold to save time)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    grid_search = GridSearchCV(
        estimator=xgb_base, 
        param_grid=param_grid, 
        scoring='accuracy', 
        cv=skf, 
        verbose=1, 
        n_jobs=-1
    )
    
    print("Executing Grid Search across 27 parameter combinations...")
    grid_search.fit(X_scaled, y_raw)
    
    print("OPTIMAL SEED-IV PARAMETERS FOUND:")
    print(grid_search.best_params_)
    print(f"Calibration Accuracy: {grid_search.best_score_ * 100:.2f}%")

if __name__ == "__main__":
    run_seed_iv_grid_search(calibration_subject=1)


=== SEED-IV Multi-Class XGBoost Grid Search (Subject 1) ===
Executing Grid Search across 27 parameter combinations...
Fitting 3 folds for each of 27 candidates, totalling 81 fits

OPTIMAL SEED-IV PARAMETERS FOUND:
{'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}
Calibration Accuracy: 85.97%



In [15]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb

def get_tuned_xgb():
    """Returns the XGBoost model tuned for SEED-IV, accelerated by GPU."""
    return xgb.XGBClassifier(
        learning_rate=0.1,      
        max_depth=6,            
        n_estimators=200,       
        objective='multi:softprob', 
        num_class=4, 
        eval_metric='mlogloss', 
        random_state=42, 
        tree_method='hist',     # <-- GPU HACK
        device='cuda'           # <-- GPU HACK
    )

def load_and_scale_data(sub_id, save_dir="./SEED_IV_FEATURES"):
    X_raw = np.load(os.path.join(save_dir, f"Sub_{sub_id}_X.npy"))
    y_raw = np.load(os.path.join(save_dir, f"Sub_{sub_id}_y.npy"))
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X_raw)
    return X_scaled, y_raw

def run_master_pipeline(total_subjects=15, epochs=30, batch_size=64, w_xgb=0.85, w_cae=0.15):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f" INITIATING MASTER PIPELINE (Hardware: {device}) ")
    
    # PHASE 1: STRATIFIED INTRA-SUBJECT (THE CEILING)
    print("\nPHASE 1: STRATIFIED INTRA-SUBJECT ")
    start_time = time.time()
    strat_accs = []
    
    for sub_id in range(1, total_subjects + 1):
        try:
            X, y = load_and_scale_data(sub_id)
            skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
            fold_accs = []
            
            for train_idx, test_idx in skf.split(X, y):
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y[train_idx], y[test_idx]
                
                # XGBoost
                xgb_model = get_tuned_xgb()
                xgb_model.fit(X_train, y_train)
                xgb_probs = xgb_model.predict_proba(X_test)
                
                # PyTorch
                X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
                y_train_t = torch.tensor(y_train, dtype=torch.long)
                X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).to(device)
                
                train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
                cae_model = MultiClassHelperCAE().to(device)
                optimizer = optim.Adam(cae_model.parameters(), lr=0.001, weight_decay=1e-5)
                
                cae_model.train()
                for _ in range(epochs):
                    for bX, by in train_loader:
                        bX, by = bX.to(device), by.to(device)
                        optimizer.zero_grad()
                        recon_x, logits = cae_model(bX)
                        loss, _, _ = seed_iv_hybrid_loss(recon_x, bX, logits, by)
                        loss.backward()
                        optimizer.step()
                        
                cae_model.eval()
                with torch.no_grad():
                    _, test_logits = cae_model(X_test_t)
                    cae_probs = F.softmax(test_logits, dim=1).cpu().numpy()
                    
                final_preds = np.argmax((w_xgb * xgb_probs) + (w_cae * cae_probs), axis=1)
                fold_accs.append(accuracy_score(y_test, final_preds) * 100)
                
            strat_accs.append(np.mean(fold_accs))
        except FileNotFoundError:
            continue
            
    print(f"Phase 1 Complete. Time: {(time.time() - start_time)/60:.1f} mins")
    print(f"Stratified Ceiling: {np.mean(strat_accs):.2f}% (±{np.std(strat_accs):.2f}%)")

    # PHASE 2: CHRONOLOGICAL (THE STARVATION TEST)
    print("\nPHASE 2: CHRONOLOGICAL INTRA-SUBJECT ")
    start_time = time.time()
    chron_accs = []
    
    for sub_id in range(1, total_subjects + 1):
        try:
            X, y = load_and_scale_data(sub_id)
            split_idx = int(len(X) * 0.80)
            X_train, X_test = X[:split_idx], X[split_idx:]
            y_train, y_test = y[:split_idx], y[split_idx:]
            
            # XGBoost
            xgb_model = get_tuned_xgb()
            xgb_model.fit(X_train, y_train)
            xgb_probs = xgb_model.predict_proba(X_test)
            
            # PyTorch
            X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
            y_train_t = torch.tensor(y_train, dtype=torch.long)
            X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).to(device)
            
            train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=batch_size, shuffle=True)
            cae_model = MultiClassHelperCAE().to(device)
            optimizer = optim.Adam(cae_model.parameters(), lr=0.001, weight_decay=1e-5)
            
            cae_model.train()
            for _ in range(epochs):
                for bX, by in train_loader:
                    bX, by = bX.to(device), by.to(device)
                    optimizer.zero_grad()
                    recon_x, logits = cae_model(bX)
                    loss, _, _ = seed_iv_hybrid_loss(recon_x, bX, logits, by)
                    loss.backward()
                    optimizer.step()
                    
            cae_model.eval()
            with torch.no_grad():
                _, test_logits = cae_model(X_test_t)
                cae_probs = F.softmax(test_logits, dim=1).cpu().numpy()
                
            final_preds = np.argmax((w_xgb * xgb_probs) + (w_cae * cae_probs), axis=1)
            chron_accs.append(accuracy_score(y_test, final_preds) * 100)
            
        except FileNotFoundError:
            continue
            
    print(f"Phase 2 Complete. Time: {(time.time() - start_time)/60:.1f} mins")
    print(f"Chronological Baseline: {np.mean(chron_accs):.2f}% (±{np.std(chron_accs):.2f}%)")

    # PHASE 3: GLOBAL LOSO (THE 'DAY 1' TEST)
    print("\n PHASE 3: GLOBAL LEAVE-ONE-SUBJECT-OUT ")
    start_time = time.time()
    
    # Load all data into memory
    all_X, all_y = {}, {}
    for i in range(1, total_subjects + 1):
        try:
            X, y = load_and_scale_data(i)
            all_X[i], all_y[i] = X, y
        except FileNotFoundError:
            continue
            
    loso_accs = []
    valid_subjects = list(all_X.keys())
    
    for test_sub in valid_subjects:
        X_test, y_test = all_X[test_sub], all_y[test_sub]
        
        X_train_list = [all_X[i] for i in valid_subjects if i != test_sub]
        y_train_list = [all_y[i] for i in valid_subjects if i != test_sub]
        X_train = np.vstack(X_train_list)
        y_train = np.concatenate(y_train_list)
        
        # XGBoost
        xgb_model = get_tuned_xgb()
        xgb_model.fit(X_train, y_train)
        xgb_probs = xgb_model.predict_proba(X_test)
        
        # PyTorch
        X_train_t = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
        y_train_t = torch.tensor(y_train, dtype=torch.long)
        X_test_t = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).to(device)
        
        # Larger batch size for the massive LOSO dataset
        train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=128, shuffle=True)
        cae_model = MultiClassHelperCAE().to(device)
        optimizer = optim.Adam(cae_model.parameters(), lr=0.001, weight_decay=1e-5)
        
        cae_model.train()
        for _ in range(epochs):
            for bX, by in train_loader:
                bX, by = bX.to(device), by.to(device)
                optimizer.zero_grad()
                recon_x, logits = cae_model(bX)
                loss, _, _ = seed_iv_hybrid_loss(recon_x, bX, logits, by)
                loss.backward()
                optimizer.step()
                
        cae_model.eval()
        with torch.no_grad():
            _, test_logits = cae_model(X_test_t)
            cae_probs = F.softmax(test_logits, dim=1).cpu().numpy()
            
        final_preds = np.argmax((w_xgb * xgb_probs) + (w_cae * cae_probs), axis=1)
        loso_accs.append(accuracy_score(y_test, final_preds) * 100)
        
    print(f"Phase 3 Complete. Time: {(time.time() - start_time)/60:.1f} mins")
    print(f"LOSO Global Baseline: {np.mean(loso_accs):.2f}% (±{np.std(loso_accs):.2f}%)")

    print("ALL BENCHMARKS COMPLETE.")

if __name__ == "__main__":
    run_master_pipeline()

 INITIATING MASTER PIPELINE (Hardware: cuda) 

PHASE 1: STRATIFIED INTRA-SUBJECT 


c:\Users\risha\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\core.py:751: UserWarning: [20:17:50] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Phase 1 Complete. Time: 25.6 mins
Stratified Ceiling: 91.28% (±3.13%)

PHASE 2: CHRONOLOGICAL INTRA-SUBJECT 
Phase 2 Complete. Time: 5.0 mins
Chronological Baseline: 33.32% (±12.48%)

 PHASE 3: GLOBAL LEAVE-ONE-SUBJECT-OUT 
Phase 3 Complete. Time: 37.5 mins
LOSO Global Baseline: 41.49% (±6.88%)
ALL BENCHMARKS COMPLETE.
